# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 agentevals==0.0.9 openevals==0.1.3

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [2]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 LangSmith 环境配置
我们需要先前往 LangSmith 的官网并进行注册登录。

登录后我们就进入了下面这个初始界面，此时我们需要找到左下角的 Setting ，然后在里面先获取新建一个 API Key。

创建完成后，我们就可以将其配置到环境变量中。除了 API_Key 以外，通常 LangSmith 的项目还需要设置是否跟踪、上传地址以及项目名称信息（这个需要自定义设置）。

In [3]:
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "your langsmith api key"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "ai-studio-evaluation"

# 2. 大模型测试

## 2.1 简介
在最新的 LangChain 文档中，添加了测试（Test）部分的内容，其主要针对的是智能体方面的测试。

日常情况下，我们通常测试的内容主要是一些实际运行的函数，其有几个非常重要的性质：
- 确定性：同样的输入 → 永远同样的输出
- 无隐式状态：除非你显式传全局变量、数据库
- 执行路径固定：不存在“运行时自己决定下一步”

这类函数测试的内容，单元测试几乎可以覆盖一切。

但智能体不一样，那么是一个简单的智能体可能就包含：
- LLM（非确定）
- Prompt（可改、影响巨大）
- Tools（可能失败、可能被“误用”）
- Memory / State（跨轮影响行为）
- 控制流（模型“自己决定”）

所以这个时候针对智能体的测试就不再只是函数，而是一个复杂的运行时决策系统。

对于工具的测试不仅仅是看智能体是否成功跑通了，而是要更进一步的考虑：
- 行为是否正确（Behavior）：参数是否正确、工具是否应该被调用。
- 轨迹是否可控（Trajectory）：调用了哪些工具、调用顺序是否正确。
- 输出是否可接受（Output）：是否符合格式要求、是否包含必须字段。
- 成本与延迟（Cost & Latency）：调用次数是否超预期。
- 稳定性与回归（Regression）：上周能过的 case，这周还能不能过。

因此对于智能体的测试，我们不能够仅仅看其是否完成正确答案，而是要更深入这个系统当中去看它是怎么得到这个答案的。这也是 LangChain Test 模块核心的设计思想，并非简单的单元测试（Unit Test），而是要更关注于集成测试（Integration Test）。

## 2.2 GenericFakeChatModel

对于大模型调用而言，由于 Transformer 架构本身的特性，因此每次生成的内容都不太稳定，这也导致假如我们希望能够测试部分内容时无法顺利实现。

因此 LangChain 团队推出了 GenericFakeChatModel 工具，其能够假装大模型的调用并返回对应格式的内容。

 对于这个 GenericFakeChatModel ，我们只需要传入一个 message 参数即可，这个 message 本质上是一个迭代器（iter），里面的元素可以是字符串或 AIMessage。每次 invoke，就消耗一个元素并输出到终端。

 ```python
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
GenericFakeChatModel(messages=iter([...]))
 ```

我们可以举一个最简单的例子，就在里面放入两个字符串，然后通过两次 model.invoke().content （返回 AIMessage 内容并获取内部 content）进行调用：

In [4]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

model = GenericFakeChatModel(
    messages=iter([
        "hello",
        "world"])
)

print(model.invoke("anything").content)
print(model.invoke("anything").content)

hello
world


此时无论传入的内容是什么，都只会输出 hello 和 world 两部分内容。所以这种方式很好的一点在于其是完全确定的，不需要依赖提示词、上下文以及其他参数。

除此之外，我们还可以精确的构造 tool_call 来进行工具调用的测试。比如说我们可以通过传入 AIMessage 的方式进行假 tool_call 的构建。

此时打印出来的就是一个带 tool_call 的 AIMessage 信息。后续基于该工具调用的信息，就可以接上真实的工具来完成测试了。

但是这个无法作为 model 放入 create_agent 中，只能用于日常的大模型调用调试中。

In [5]:
from langchain_core.messages import AIMessage 
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

fake_model = GenericFakeChatModel(
    messages=iter([
        AIMessage(
            content="",
            tool_calls=[
                {
                    "id": "call_1",
                    "name": "foo",
                    "args": {"bar": "baz"},
                }
            ]
        ),
        "final answer"
    ])
)

print(fake_model.invoke("anything"))

content='' additional_kwargs={} response_metadata={} id='lc_run--019ba303-5a85-7762-9d9b-851db4b1c49e-0' tool_calls=[{'name': 'foo', 'args': {'bar': 'baz'}, 'id': 'call_1', 'type': 'tool_call'}] invalid_tool_calls=[]


## 2.3 行为轨迹评估器

在智能体的测试中，最重要的是关注其 Agent Trajectory（行为轨迹）。也就是说大模型最终回答对不对远远不够，我们必须测试“它是怎么做出这个回答的”？即便大模型给的回复最终是一样的，但是实现过程天差地别。

可能同样是询问一个天气问题，两个 agent 给出了类似的回复，但是Agent A 的行为轨迹是：
- 直接调用 get_weather(city="Beijing")
- 返回结果
- 总结

而 Agent B 的行为轨迹是：
- 调用 search("Beijing weather")
- 再调用 get_weather(city="Beijing")
- 又调用一次 get_weather(city="Beijing")
- 再总结

所以虽然输出是一样的，但是成本、延迟、行为安全都不太一样，所以我们必须对其颞部进行监控。

那这里所谓的 Agent Trajectory （行为轨迹）本质上就是一次 Agent 执行过程中，所有 Message 的有序序列，这里通常包括：
- HumanMessage（用户输入）
- AIMessage（模型回复）
- AIMessage 中的 tool_calls
- ToolMessage（工具返回）
- 最终 AIMessage（总结）

那在智能体测试中主要也是测试行为链条是否和我们想象的是一样的。

这些测试都是需要真实的模型来进行测试，Fake Model 是“演不出来的”。而在 LangChain 中专门推出了 agentevals 库来完成这方面的工作。

Agent Trajectory （行为轨迹）主要有两种评估的模式：
- 第一种是规则式（Trajectory Match Evaluator），其主要评估方式是：
    - 我提前知道“正确行为长什么样”
    - 拿实际轨迹逐步对比
- 该方法的优点是：
    - 稳定
    - 可重复
    - 不额外消耗 LLM
    - 非常适合持续集成
- 但是缺点是行为必须相对明确才能正确的评估。

- 另外一种是评审式（LLM-as-Judge Evaluator），其主要的评估方式是：
    - 不写死规则
    - 用另一个 LLM 来“评判是否合理”
- 其优点在于：
    - 灵活
    - 能评估“质量”“是否绕路”
- 而缺点是：
    - 不确定
    - 有成本
    - 不适合高频持续集成

### 2.3.1 Trajectory Match Evaluator
对于流程级的测试中，我们需要准备的有三部分内容：
- inputs：用户输入
- outputs：Agent 实际跑出来的 messages
- reference_outputs：你“认为正确的行为轨迹”

Trajectory Match Evaluator 做的事是看基于 inputs 输入获取的 outputs 是否满足 reference_outputs 的约束。

虽然内容上可能不一样，但是回复的整体行为是一样的就可以。

在 LangChain 中提供了四种 Trajectory Match 模式：

| 模式        | 行为约束                     | 适合场景                     |
|-------------|------------------------------|------------------------------|
| **strict**      | 顺序、工具调用完全一致           | 合规 / 审批 / 强流程            |
| **unordered**  | 工具集合一致，顺序不重要         | 信息收集                     |
| **subset**     | 不允许多余工具                 | 权限 / 安全                   |
| **superset**   | 至少做这些事                 | 最低行为保障                 |


#### 2.3.1.1 Strict Match
在这个模式下，每一步的工作调用以及顺序都需要和我们设定的目标是一致的，因此非常适合一些要求非常严格的场景下。

首先我们需要导入 create_trajectory_match_evaluator 并将模式设置为 strict：

In [6]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

evaluator = create_trajectory_match_evaluator(trajectory_match_mode="strict")  

然后我们需要准备前面提到的三部分内容：
- inputs：用户输入
- outputs：Agent 实际跑出来的 messages
- reference_outputs：你“认为正确的行为轨迹”

比如这里我们定义了一个函数，将输入设置为 "What's the weather in San Francisco?"。输出设置为调用输入后的结果 result：

In [7]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from agentevals.trajectory.match import create_trajectory_match_evaluator
from langchain_community.chat_models import ChatTongyi
import os

model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

@tool
def get_weather(city: str):
    """Get weather information for a city."""
    return f"It's 75 degrees and sunny in {city}."

agent = create_agent(model, tools=[get_weather])

evaluator = create_trajectory_match_evaluator(  
    trajectory_match_mode="strict",  
)  

def test_weather_tool_called_strict():
    result = agent.invoke({
        "messages": [HumanMessage(content="What's the weather in San Francisco?")]
    })

    reference_trajectory = [
        HumanMessage(content="What's the weather in San Francisco?"),
        AIMessage(content="", tool_calls=[
            {"id": "call_1", "name": "get_weather", "args": {"city": "San Francisco"}}
        ]),
        ToolMessage(content="It's 75 degrees and sunny in San Francisco.", tool_call_id="call_1"),
        AIMessage(content="The weather in San Francisco is 75 degrees and sunny."),
    ]

    evaluation = evaluator(
        outputs=result["messages"],
        reference_outputs=reference_trajectory
    )
    print(evaluation)
    assert evaluation["score"] is True
    
test_weather_tool_called_strict()

{'key': 'trajectory_strict_match', 'score': True, 'comment': None, 'metadata': None}


参考的输出则是用一个列表将整个计划中的流程按顺序进行了设置。

最后将输出结果中的 messages （也就是调用的历史记录）与参考的输出一同传入到 evalutor 获取结果。

正常情况下，evaluator 在运行完结果后会返回一个字典，里面包含四个字段：
- key：测试的类型是 trajectory_strict_match
- score：测试的结果，如何没问题就是 True，有问题就是 False
- comment：评论，该模式下返回的内容为 None 。一般是 LLM-as-Judge Evaluator 才会存在
- metadata：元数据

```python
    {
        'key': 'trajectory_strict_match',
        'score': True,
        'comment': None,
        'metadata': None
    }
```

在当前假工具的情况下，这种一般不会出错，但是严格模式下主要审查的点有几个：
- 消息数量是否一致，不一致直接判定为不通过
- 消息角色是否一一对应，输出结果与预期结果的角色不匹配则不通过
- 是否同时存在或同时不存在 tool_calls，仅一方存在则不通过
- tool_calls 数量是否一致，数量不一致则不通过
- tool_calls 中的工具名称是否一致，存在不匹配则不通过
- tool_calls 的参数值是否一致，参数不一致同样判定为不通过

基于以上规则，可以看到下面的示例中，虽然绝大部分的内容是一致的，但就是因为 tool_calls 里的内容不一样所以返回的 score 就是 False：

In [8]:
import json
from agentevals.trajectory.match import create_trajectory_match_evaluator

outputs = [
    {"role": "user", "content": "What is the weather in San SF?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_weather", "arguments": json.dumps({"city": "San Francisco"})}},
        {"function": {"name": "accuweather_forecast", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "It's 80 degrees and sunny in SF."},
    {"role": "assistant", "content": "The weather in SF is 80 degrees and sunny."}]

reference_outputs = [
    {"role": "user", "content": "What is the weather in San Francisco?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_weather", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "It's 80 degrees and sunny in San Francisco."},
    {"role": "assistant", "content": "The weather in SF is 80˚ and sunny."},
]

evaluator = create_trajectory_match_evaluator(trajectory_match_mode="strict")

result = evaluator(outputs=outputs, reference_outputs=reference_outputs)

print(result)


{'key': 'trajectory_strict_match', 'score': False, 'comment': None, 'metadata': None}


但是假如两者的内容差别并不大，只不过是大小写的区别，比如 "san francisco" 和 "San Francisco" ，这个情况下假如我们还严格设定其为不一样的话，显然不太合理。
因此在评估器设置时，我们可以对 tool_args_match_mode 进行设置，比如：

In [9]:
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
    tool_args_match_mode="exact")  # 模型情况

tool_args_match_mode 有四种模式：
- exact（默认）：参数必须完全一致。该模式校验最为严格，只有当工具调用的参数结构和取值都高度稳定、且不希望 LLM 进行任何“自由补充或省略”时才适合使用。
- ignore：只要工具名称一致即可视为匹配。该模式完全忽略参数内容，仅关注工具调用是否发生以及调用顺序是否正确。
- subset：输出参数是参考参数的子集。
- superset：输出参数是参考参数的超集。

除了这四种模式以外，还有一种更高级的方法 tool_args_match_overrides ，其允许对某些工具单独定规则，其优先级比起前面四种模式都要高。
比如说最开始讲的 "san francisco" 和 "San Francisco" 例子，即便我们使用前面四种模式都是无法解决的，那这个时候我们可以自己来制定规则，比如：

In [10]:
tool_args_match_overrides={"get_weather": lambda x, y: x["city"].lower() == y["city"].lower()}

此时就可以通过将 city 参数都转变为小写的，那这个时候再进行对比就会返回 True 了。

当然使用函数是比较复杂的用法，比较简单的用法比如指定某个工具的对应参数用某种模式的话，可以通过：

In [11]:
tool_args_match_overrides={"get_weather": "ignore"}

在这个情况下，get_weather 这个工具就完全不会考虑参数里面具体内容的审查了。又比如我们只对比一部分的字段：

In [12]:
tool_args_match_overrides={"get_weather": ["city"]}

那此时只要这里 city 的信息一样即可，其他参数不一样也没关系。

那 overrides 方法可以和正常方法进行一起写，系统会将 overrider 方法有的内容覆盖原有的方法，其他的都保持不变，比如：

In [13]:
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
    tool_args_match_mode="exact",  # Default value
    tool_args_match_overrides={
        "get_weather": lambda x, y: x["city"].lower() == y["city"].lower()
    }
)

此时审查的话，"san francisco" 和 "San Francisco" 即便不同也会 score 返回为 True 了。当然两段代码目前的问题不仅仅只是该信息不同，还有其他很多不一样的地方，所以返回的内容还是 False。

In [14]:
result = evaluator(outputs=outputs, reference_outputs=reference_outputs)

print(result)

{'key': 'trajectory_strict_match', 'score': False, 'comment': None, 'metadata': None}


#### 2.3.1.2 Unordered Match
相比于 Strict 的形式，unordered 不再“逐步对齐消息”，而是“整体集合判断”。其关心的思路与前面要求一一对应的思路不同：
- ❌ 不关心 message 数量
- ❌ 不关心 message 顺序
- ❌ 不关心 tool_call 出现在哪一条 message
- ✅ 只关心“你有没有调用过这些工具（按名字 + 参数规则）”

比如在下面的示例中，在 outputs 里虽然是分开两次进行工具的调用，而在 reference_outputs 里则是一次性将两个工具进行调用：

In [15]:
inputs = {}
outputs = [
    {"role": "user", "content": "What is the weather in SF and is there anything fun happening?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_weather", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "It's 80 degrees and sunny in SF."},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_fun_activities", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "Nothing fun is happening, you should stay indoors and read!"},
    {"role": "assistant", "content": "The weather in SF is 80 degrees and sunny, but there is nothing fun happening."},
]


假如这在 Strict 模式下就会判定为 False 了。但是在 Unordered 模式下由于调用的内容其实是一样的，所以就还是会返回 True：

In [16]:
reference_outputs = [
    {"role": "user", "content": "What is the weather in SF and is there anything fun happening?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"function": {"name": "get_fun_activities", "arguments": json.dumps({"city": "San Francisco"})}},
        {"function": {"name": "get_weather", "arguments": json.dumps({"city": "San Francisco"})}},
    ]},
    {"role": "tool", "content": "Nothing fun is happening, you should stay indoors and read!"},
    {"role": "tool", "content": "It's 80 degrees and sunny in SF."},
    {"role": "assistant", "content": "In SF, it's 80˚ and sunny, but there is nothing fun happening."},
]

evaluator = create_trajectory_match_evaluator(trajectory_match_mode="unordered")
print(evaluator(outputs=outputs, reference_outputs=reference_outputs))

{'key': 'trajectory_unordered_match', 'score': True, 'comment': None, 'metadata': None}


所以可以看出来，主要 unordered 主要审查的是工具是否正确调用了，而不去考虑顺序是否一致等等的内容，因此比较适合信息收集、搜索以及多来源查询等应用场景。在这些场景中，“有没有查天气”比“城市字符串是否完全一致”更重要。

#### 2.3.1.3 Subset Match

和 Unordered 模式类似，Subset 也是只比较 tool_call 多重集合。

但是 Subset 模式的核心区别在于“是否越权”。

当 reference 要求调用 get_weather 和 get_events 时，如果 Agent 实际只调用了 get_weather，虽然少执行了一部分操作，但并没有调用 reference 之外的工具，因此仍被视为合规，最终返回 True。

甚至在更极端的情况下，reference 要求调用 get_weather 和 get_events，而 Agent 实际没有调用任何工具，Subset 依然会返回 True。这是因为该模式并不关心“是否做完”，而只关注“是否越界”。

但一旦 Agent 出现了额外调用的情况，例如：
- reference 要求 get_weather、get_events，而 Agent 实际调用了 get_weather、get_events、search
- reference 要求 get_weather、get_events，而 Agent 实际调用了 get_weather、get_events、get_weather

只要存在 reference 未允许的额外工具调用，Subset 都会直接判定为 False。

可以看出，Subset 模式允许“少做事”，但绝不允许“多做事”。

总的来说，Subset 方法比较适合以下场景：
- 权限 / 安全：Agent 只能访问某些工具且多一个 API 调用都不行
- 成本控制：不允许多次搜索也不允许调用昂贵模型
- 教学 / 作业：学生只能用指定工具从而防止“多调接口投机取巧”

Subset 的 evaluator 的创建方法也很简单，就是修改名称即可：

In [17]:
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="subset", 
)

#### 2.3.1.4 Superset Match
Superset 与 Unordered / Subset 一样，仅关注工具调用本身，而不关心上下文细节。

但在判定规则上，Superset 与 Subset 正好相反。

Subset 允许少做事，但不允许多做事；而 Superset 则允许多做事，但不允许少做事。

例如，当 reference 要求调用 get_weather 和 get_events 时：
- 若 Agent 实际调用了 get_weather、get_events、get_weather，在 superset 模式下仍会判定为 True，因为所有必需的工具都已覆盖。
- 但如果 Agent 只调用了其中一部分，则会直接判定为 False。

需要注意的是，调用次数同样是约束的一部分。如 reference 要求调用两次 get_weather，而 Agent 实际只调用了一次，即便工具名称匹配，superset 依然会返回 False。

此外，还有一个特殊情况，即当 reference 本身不要求任何工具调用时（即为空），无论 Agent 是否调用工具，superset 都会返回 True。

Superset 的 evaluator 的创建方法也很简单，就是修改名称即可：

In [18]:
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="superset", 
)

总体而言，superset 的设计目标并不是限制 Agent 的发挥，而是确保关键步骤没有被遗漏。

因此，它最适合用于以下场景：
- 必做步骤校验：是否至少进行过一次检索、一次校验、一次数据库访问
- Planner / Research Agent：允许多轮探索，但必须覆盖核心信息源
- 教学中的“最低合格线”：不要求解法唯一，但关键步骤必须出现

这种模式本质上是在评估：“该做的事情，有没有做过。”

#### 2.3.1.5 模式对比

因此对比三种审查工具信息的模式，可以得到以下的表格：

| 模式        | 数学关系              | 实际表达                     |
|-------------|-----------------------|------------------------------|
| **superset**   | A ⊇ R                 | 你至少把我要求的都做了        |
| **subset**     | A ⊆ R                 | 你没做我不允许的事            |
| **unordered**  | A ⊇ R 且 R ⊇ A        | 你做的和我期望的一模一样      |


### 2.3.2 LLM-as-Judge Evaluator

除了通过规则的方式去评价以外，其实我们还可以考虑通过大模型来进行评价整个流程是否正常合理。

通过规则评判的前提是你能“提前写出规则”，但现实里，经常是下面这种情况：
- Agent 的推理步骤会变
- 工具调用顺序会变
- 有时候多一步，有时候少一步
- 但整体行为仍然是“合理的”

这时规则式 matcher 会失败，但人类会说“这没问题”。因此为了解决这一类的问题，我们可以通过更专业或者说更强大的模型模拟“人类评审员”。

在 LangChain 中内置了一段系统提示词完成该部分任务：

In [19]:
TRAJECTORY_ACCURACY_PROMPT = """You are an expert data labeler.
Your task is to grade the accuracy of an AI agent's internal trajectory.
<Rubric>
  An accurate trajectory:
  - Makes logical sense between steps
  - Shows clear progression
  - Is relatively efficient, though it does not need to be perfectly efficient
</Rubric>
First, try to understand the goal of the trajectory by looking at the input
(if the input is not present try to infer it from the content of the first message),
as well as the output of the final message. Once you understand the goal, grade the trajectory
as it relates to achieving that goal.
Grade the following trajectory:
<trajectory>
{outputs}
</trajectory>
"""

所以从这段提示词中可以看到，LLM 会被要求判断：
- Agent 是否正确理解了用户问题
- 是否调用了合适的工具
- 是否有明显多余或错误的工具调用
- 最终回答是否和工具结果一致
- 整体执行路径是否高效、自然

所以这是“语义层面”的判断，而不是结构层面的判断！当然这部分提示词我们可以自行替换，但是一般情况下默认的就足够了。

为了能够实现让大模型进行判断，我们首先需要先定义一个判断的大模型，比如这里我们还是使用 ChatTongyi 来进行完成，然后提示词就使用前面展示的提示词：

In [20]:
from agentevals.trajectory.llm import create_trajectory_llm_as_judge, TRAJECTORY_ACCURACY_PROMPT

model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")
evaluator = create_trajectory_llm_as_judge(prompt=TRAJECTORY_ACCURACY_PROMPT, judge=model)

然后我们就可以来设置一个函数来传入 outputs 的内容并且直接让大模型评判 outputs 的内容是否合理（不需要传入 references 了！）：

In [22]:
def test_trajectory_quality():
    result = agent.invoke({"messages": [HumanMessage(content="What's the weather in Seattle?")]})
    evaluation = evaluator(outputs=result["messages"])
    print(evaluation)
    assert evaluation["score"] is True
test_trajectory_quality()

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'key': 'trajectory_accuracy', 'score': True, 'comment': "The trajectory makes logical sense as it starts with the user asking about the weather in Seattle, and the AI agent correctly calls a function to retrieve the weather information. The progression is clear, moving from the initial question to the function call and then to the final response. The process is efficient as it directly addresses the user's query without unnecessary steps. Thus, the score should be: true.", 'metadata': None}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


当然这里演示的是没有 references 的情况下，假如我们要加上 references 答案的话一样也是 ok 的，只不过我们需要先更换一下模型提示词为 TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE，也就是：

In [23]:
TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE = """You are an expert data labeler.
Your task is to grade the accuracy of an AI agent's internal trajectory.
<Rubric>
  An accurate trajectory:
  - Makes logical sense between steps
  - Shows clear progression
  - Is relatively efficient, though it does not need to be perfectly efficient
  - Is semantically equivalent to the provided reference trajectory
</Rubric>
Based on the following reference trajectory:
<reference_trajectory>
{reference_outputs}
</reference_trajectory>
Grade this actual trajectory:
<trajectory>
{outputs}
</trajectory>
"""

这里其实也就是把参考答案加进去了而已，所以模型和之前是一样的：

In [24]:
from agentevals.trajectory.llm import TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE

evaluator = create_trajectory_llm_as_judge(judge=model,
    prompt=TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,)

然后呢在评估的时候我们也需要更新一下，不再单纯传入的是 outputs 的内容了，而是要把参考的路径也给进去，比如：

```python
evaluation = judge_with_reference(outputs=result["messages"],
    reference_outputs=reference_trajectory)
```

在创建 create_trajectory_llm_as_judge 时（直接使用的是 openevals 仓库中的 create_llm_as_judge 方法），除了上面提到的 prompt 和 model 两个参数，其实还有其他的参数，包括：

```python
scorer = create_trajectory_llm_as_judge(
    prompt=prompt,              # 评审规则 +任务说明
    judge=judge,                # 谁来评？
    model=model,                # 用哪个模型来评？
    continuous=continuous,      # 打连续分还是二值？
    choices=choices,            # 分数档位
    use_reasoning=use_reasoning,# 要不要解释
    few_shot_examples=few_shot_examples,  # 示例教学
)
```

## 2.5 总结

总的来说，传统的软件测试，关注的是“输入—输出是否一致”，而智能体系统测试，关注的则是“它是如何做出这个输出的”。当系统中引入了 LLM、工具调用、状态记忆与动态控制流后，行为本身就成为了核心资产。

LangChain Test 模块与 agentevals 的出现，正是为了补上这一块长期缺失的工程能力，让智能体的行为轨迹变得可观测、可断言、可回归。

在实际工程中，这套体系并不是用来“挑模型毛病”的，而是帮助开发者区分清楚：哪些问题是模型本身的不确定性，哪些问题其实是我们自己系统设计的漏洞。

通过 Unit Test 固定确定性逻辑，再通过 Integration Test 约束真实行为轨迹，再结合规则式 Match 与 Judge 式评估，我们终于可以像对待传统软件一样，对智能体系统建立起一套可维护、可演进的质量保障机制。